# LCG Cluster Analysis (AWS Embeddings)

This notebook runs the same AWS embedding + PCA + KMeans workflow, focused on the LCG subset table (`PRODUCTS_LCG`).

It performs one embedding pass, then evaluates two K ranges:
- L4-focused: `k in [3, 5, 8, 10, 12, 15]`
- L5-exploration: `k in [20, 30, 40, 50]`

It reports local silhouette peaks per PCA setting for each range.

In [1]:
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.metrics import silhouette_score

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "product_classifier_utils.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "product_classifier_utils.py").exists():
    raise FileNotFoundError("Could not locate project root containing product_classifier_utils.py")

import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from product_classifier_utils import (  # noqa: E402
    build_product_text,
    embed_texts_from_cache,
    get_bedrock_client,
    get_snowflake_session,
    load_product_data,
    stable_text_hash,
)

In [2]:
# Core run configuration
TABLE = "SNOWFLAKE_LEARNING_DB.SMCMAHON_PRODUCTS.PRODUCTS_LCG"
LABEL_COLUMN = "PARENT_3_CATEGORY"
MIN_CATEGORY_COUNT = 1
ONLY_PRICED = False
ROW_LIMIT = None

# Embedding + cache settings
AWS_PROFILE = "staging.admin"
AWS_REGION = "us-east-1"
MODEL_ID = "amazon.titan-embed-text-v1"
MAX_WORKERS = 16
CHECKPOINT_EVERY = 2500
MAX_RETRIES = 5
CACHE_PATH = PROJECT_ROOT / "artifacts/cache/embedding_cache.pkl"

# PCA + K exploration settings
PCA_COMPONENTS_TO_TRY = [None, 128, 256, 512, 768]
K_VALUES_L4 = [3, 5, 8, 10, 12, 15]
K_VALUES_L5 = [20, 30, 40, 50]
K_VALUES_ALL = sorted(set(K_VALUES_L4 + K_VALUES_L5))
SAMPLE_SIZE_FOR_SILHOUETTE = 30000
RANDOM_STATE = 42

# Output settings
LOCAL_OUTPUT_DIR = PROJECT_ROOT / "artifacts" / "analysis"
RUN_TAG = "lcg_aws_cluster_experiment"
SAVE_TO_SNOWFLAKE_TABLE = False
OUTPUT_TABLE_NAME = "PROPOSED_LCG_AWS"

# Optional LLM labeling
RUN_LLM_LABELING = True
LLM_MODEL_ID = "anthropic.claude-3-5-sonnet-20240620-v1:0"
LLM_MAX_CATEGORIES = 12
LLM_TEMPERATURE = 0.2

In [3]:
sf_session = get_snowflake_session()
df = load_product_data(
    session=sf_session,
    table=TABLE,
    label_column=LABEL_COLUMN,
    min_category_count=MIN_CATEGORY_COUNT,
    row_limit=ROW_LIMIT,
)

if ONLY_PRICED:
    df = df[df["PRICING_STATUS_C"].astype(str).str.upper() == "PRICED"].reset_index(drop=True)

required = {"PRODUCT_ID", "PRODUCT_NAME", "DESCRIPTION", "PRICING_STATUS_C", "LIST_PRICE_C"}
missing = sorted(required - set(df.columns))
if missing:
    raise ValueError(f"Missing required columns for text embedding: {missing}")

print(f"Loaded rows: {len(df):,}")
print(df[["PRODUCT_ID", "PRICING_STATUS_C", LABEL_COLUMN]].head(3))

Initiating login request with your identity provider. Press CTRL+C to abort and try again...
Going to open: https://scienceexchange.okta.com/app/snowflake/exktzixqmxM7e7kdB697/sso/saml?SAMLRequest=lZJfT9swFMW%2FSuQ9J3ZCaYPVFhWqlkzAurYgjTcvuW2tJHbwdUjKp5%2FTPxN7AGlvkXOOf8f33OF1WxbeGxiUWo1IGDDigUp1JtV2RJ7WMz8mHlqhMlFoBSOyByTX4yGKsqj4pLY7tYTXGtB67iKFvPsxIrVRXAuUyJUoAblN%2BWrycM%2BjgHGBCMY6HDlZMpSOtbO24pQ2TRM0F4E2Wxoxxii7ok7VSb6RD4jqa0ZltNWpLs6W1r3pE0RIWa9DOIUjLE7GG6mOI%2FiK8vsoQn63Xi%2F8xY%2FVmniT8%2BtutcK6BLMC8yZTeFreHwOgS1Dl6eWgdxkHNfog0PphgEo3m0LkkOqyqq27NnBfdAMZLfRWumEl0xGpcpnVSvae76KkXrZ5M3%2BPkuWvebzaCf29ZT0z30%2Fil%2Frnk57N9mlKvOdztVFXbYJYQ6K6Qq07YlHfZxc%2B66%2FDmIchZ2HQi6MX4k1doVIJe3CeU2Mq3ZAA2nQn1BYCnVtxCCmqiv7NT6HN7btsX8v2YQCDPLvpXw0ooqZdb%2BS4OvwQxIz%2FeyBD%2BtF%2BWsNH10wyXehCpntvpk0p7OfFhUF4OJGZvzlIOZRCFpMsM4DoCiwK3dwaENZtuzU1EDo%2BUv%2Fd9%2FEf&RelayState=ver%3A3-hint%3A9376132672794866-ETMsDgAAAZzEWITqABRBRVMvQ0JDL1BLQ1M1UGFkZGluZwEAABAAENYZe9aP1qc9ZttzdLm4SYwAAACgrECfEclRACw

In [4]:
def load_cache(path: Path) -> dict:
    if not path.exists():
        return {}
    with path.open("rb") as f:
        obj = pickle.load(f)
    if not isinstance(obj, dict):
        raise ValueError(f"Cache file {path} is not a dict")
    return obj


def save_cache(path: Path, cache: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("wb") as f:
        pickle.dump(cache, f)


text_series = build_product_text(df)
texts = text_series.tolist()
text_hashes = [stable_text_hash(t) for t in texts]

cache = load_cache(CACHE_PATH)
cache_before = len(cache)
client = get_bedrock_client(profile_name=AWS_PROFILE, region=AWS_REGION)


def checkpoint_cb(cache_obj: dict, processed_count: int) -> None:
    save_cache(CACHE_PATH, cache_obj)
    print(f"Checkpointed {processed_count:,} new embeddings. Cache size={len(cache_obj):,}")


X = embed_texts_from_cache(
    texts=texts,
    text_hashes=text_hashes,
    cache=cache,
    client=client,
    model_id=MODEL_ID,
    show_progress=True,
    max_workers=MAX_WORKERS,
    checkpoint_every=CHECKPOINT_EVERY if CHECKPOINT_EVERY > 0 else None,
    on_checkpoint=checkpoint_cb if CHECKPOINT_EVERY > 0 else None,
    max_retries=MAX_RETRIES,
)

save_cache(CACHE_PATH, cache)
print(f"Embedding matrix shape: {X.shape}")
print(f"Cache size: {cache_before:,} -> {len(cache):,}")

df["TEXT_TO_EMBED"] = text_series

Embedding missing texts:   1%|          | 2950/432770 [00:35<2:13:14, 53.77it/s] 

Checkpointed 2,500 new embeddings. Cache size=1,164,104


Embedding missing texts:   1%|          | 5262/432770 [01:43<3:29:51, 33.95it/s] 

Checkpointed 5,000 new embeddings. Cache size=1,166,604


Embedding missing texts:   2%|▏         | 7775/432770 [02:57<3:09:36, 37.36it/s] 

...

Checkpointed 430,000 new embeddings. Cache size=1,591,604


Embedding missing texts: 100%|█████████▉| 432501/432770 [6:29:39<00:34,  7.88it/s]

Checkpointed 432,500 new embeddings. Cache size=1,594,104


Embedding missing texts: 100%|██████████| 432770/432770 [6:29:40<00:00, 18.51it/s]


Embedding matrix shape: (450405, 1536)
Cache size: 1,161,604 -> 1,594,374


In [ ]:
def find_local_maxima(score_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for pca_label, grp in score_df.groupby("pca"):
        g = grp.sort_values("k").reset_index(drop=True)
        for i in range(len(g)):
            left = g.loc[i - 1, "silhouette"] if i > 0 else -np.inf
            cur = g.loc[i, "silhouette"]
            right = g.loc[i + 1, "silhouette"] if i < len(g) - 1 else -np.inf
            if cur >= left and cur >= right:
                rows.append(g.loc[i].to_dict())
    if not rows:
        return pd.DataFrame(columns=score_df.columns)
    return pd.DataFrame(rows).sort_values("silhouette", ascending=False).reset_index(drop=True)


rng = np.random.default_rng(RANDOM_STATE)
sample_n = min(SAMPLE_SIZE_FOR_SILHOUETTE, len(X))
sample_idx = rng.choice(len(X), size=sample_n, replace=False)

rows = []
transformed_by_pca = {}
pca_model_by_label = {}

for pca_components in PCA_COMPONENTS_TO_TRY:
    if pca_components is None:
        X_eval = X
        pca_label = "none"
        explained = np.nan
        pca_model = None
    else:
        pca_model = PCA(n_components=pca_components, random_state=RANDOM_STATE)
        X_eval = pca_model.fit_transform(X).astype(np.float32)
        pca_label = str(pca_components)
        explained = float(np.sum(pca_model.explained_variance_ratio_))

    transformed_by_pca[pca_label] = X_eval
    pca_model_by_label[pca_label] = pca_model

    X_sample = X_eval[sample_idx]
    for k in K_VALUES_ALL:
        km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init="auto", max_iter=300)
        labels = km.fit_predict(X_sample)
        sil = silhouette_score(X_sample, labels)
        rows.append(
            {
                "pca": pca_label,
                "pca_components": pca_components,
                "k": int(k),
                "silhouette": float(sil),
                "explained_variance_sum": explained,
                "range": "l4" if k in K_VALUES_L4 else "l5",
            }
        )

results_df = pd.DataFrame(rows).sort_values("silhouette", ascending=False).reset_index(drop=True)
results_df.head(20)

In [ ]:
results_l4 = results_df[results_df["range"] == "l4"].copy()
results_l5 = results_df[results_df["range"] == "l5"].copy()

local_max_l4 = find_local_maxima(results_l4)
local_max_l5 = find_local_maxima(results_l5)

print("Top L4-range results:")
display(results_l4.head(12))

print("\nLocal maxima in L4-range:")
display(local_max_l4.head(12))

print("\nTop L5-range results:")
display(results_l5.head(12))

print("\nLocal maxima in L5-range:")
display(local_max_l5.head(12))

In [ ]:
# Choose final configs (defaults to best silhouette in each range)
BEST_L4 = local_max_l4.iloc[0] if len(local_max_l4) else results_l4.iloc[0]
BEST_L5 = local_max_l5.iloc[0] if len(local_max_l5) else results_l5.iloc[0]

FINAL_PCA_L4 = str(BEST_L4["pca"])
FINAL_K_L4 = int(BEST_L4["k"])
FINAL_PCA_L5 = str(BEST_L5["pca"])
FINAL_K_L5 = int(BEST_L5["k"])

print("Chosen L4 config:", {"pca": FINAL_PCA_L4, "k": FINAL_K_L4, "silhouette": float(BEST_L4["silhouette"])})
print("Chosen L5 config:", {"pca": FINAL_PCA_L5, "k": FINAL_K_L5, "silhouette": float(BEST_L5["silhouette"])})

# Fit final models on full set
X_l4 = transformed_by_pca[FINAL_PCA_L4]
X_l5 = transformed_by_pca[FINAL_PCA_L5]

km_l4 = KMeans(n_clusters=FINAL_K_L4, random_state=RANDOM_STATE, n_init="auto", max_iter=300)
km_l5 = KMeans(n_clusters=FINAL_K_L5, random_state=RANDOM_STATE, n_init="auto", max_iter=300)

df["CLUSTER_L4"] = km_l4.fit_predict(X_l4).astype("int32")
df["CLUSTER_L5_EXPLORATORY"] = km_l5.fit_predict(X_l5).astype("int32")

display(df[["CLUSTER_L4", "CLUSTER_L5_EXPLORATORY"]].describe())

In [ ]:
# Cluster interpretation terms for both L4 and L5 cluster outputs.
DOMAIN_STOPWORDS = {
    "price", "pricing", "status", "quote", "quoted", "description", "insert",
    "item", "items", "product", "products", "unit", "units",
}
STOPWORDS = list(ENGLISH_STOP_WORDS.union(DOMAIN_STOPWORDS))

vectorizer = TfidfVectorizer(
    max_features=20000,
    stop_words=STOPWORDS,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.8,
)

tfidf_matrix = vectorizer.fit_transform(df["TEXT_TO_EMBED"])
feature_names = np.array(vectorizer.get_feature_names_out())


def top_terms_per_cluster(tfidf, dataf, cluster_col, features, top_n=15):
    out = {}
    for c in sorted(dataf[cluster_col].unique()):
        mask = (dataf[cluster_col] == c).values
        centroid = np.asarray(tfidf[mask].mean(axis=0)).ravel()
        top_idx = centroid.argsort()[::-1][:top_n]
        out[int(c)] = features[top_idx].tolist()
    return out


cluster_keywords_l4 = top_terms_per_cluster(tfidf_matrix, df, "CLUSTER_L4", feature_names, top_n=15)
cluster_keywords_l5 = top_terms_per_cluster(tfidf_matrix, df, "CLUSTER_L5_EXPLORATORY", feature_names, top_n=15)

print("Sample L4 cluster terms:")
display(dict(list(cluster_keywords_l4.items())[:3]))

print("Sample L5 cluster terms:")
display(dict(list(cluster_keywords_l5.items())[:3]))

In [ ]:
out_dir = LOCAL_OUTPUT_DIR / RUN_TAG
out_dir.mkdir(parents=True, exist_ok=True)

results_df.to_csv(out_dir / "pca_k_silhouette_all.csv", index=False)
results_l4.to_csv(out_dir / "pca_k_silhouette_l4_range.csv", index=False)
results_l5.to_csv(out_dir / "pca_k_silhouette_l5_range.csv", index=False)
local_max_l4.to_csv(out_dir / "local_maxima_l4_range.csv", index=False)
local_max_l5.to_csv(out_dir / "local_maxima_l5_range.csv", index=False)

(df[["PRODUCT_ID", LABEL_COLUMN, "PRICING_STATUS_C", "CLUSTER_L4", "CLUSTER_L5_EXPLORATORY", "TEXT_TO_EMBED"]]
   .to_csv(out_dir / "lcg_cluster_assignments.csv", index=False))

pd.DataFrame(
    [{"cluster_l4": c, "top_terms": ", ".join(terms)} for c, terms in cluster_keywords_l4.items()]
).to_csv(out_dir / "lcg_cluster_l4_top_terms.csv", index=False)

pd.DataFrame(
    [{"cluster_l5": c, "top_terms": ", ".join(terms)} for c, terms in cluster_keywords_l5.items()]
).to_csv(out_dir / "lcg_cluster_l5_top_terms.csv", index=False)

run_config = {
    "table": TABLE,
    "label_column": LABEL_COLUMN,
    "min_category_count": MIN_CATEGORY_COUNT,
    "only_priced": ONLY_PRICED,
    "row_limit": ROW_LIMIT,
    "embedding_model_id": MODEL_ID,
    "final_l4": {"pca": FINAL_PCA_L4, "k": FINAL_K_L4},
    "final_l5": {"pca": FINAL_PCA_L5, "k": FINAL_K_L5},
    "random_state": RANDOM_STATE,
}
with (out_dir / "run_config.json").open("w", encoding="utf-8") as f:
    json.dump(run_config, f, indent=2)

if SAVE_TO_SNOWFLAKE_TABLE:
    snow_df = sf_session.create_dataframe(df)
    snow_df.write.mode("overwrite").save_as_table(OUTPUT_TABLE_NAME)
    print(f"Wrote Snowflake table: {OUTPUT_TABLE_NAME}")

print(f"Saved artifacts to: {out_dir}")

In [ ]:
# Optional: LLM-assisted naming for both L4 and L5 cluster ranges.
if RUN_LLM_LABELING:
    def suggest_cluster_labels_with_bedrock(cluster_terms: dict, range_name: str):
        prompt = f"""
You are helping design product taxonomy labels for a Laboratory Consumables & Glassware subset.
Given cluster keyword lists from the {range_name} clustering output, propose concise category labels.

Rules:
- Return strict JSON only.
- Use clear business-facing labels.
- Keep labels distinct and reusable.
- Prefer <= {LLM_MAX_CATEGORIES} categories when reasonable.

Input:
{json.dumps(cluster_terms, indent=2)}

Return JSON object with keys:
- cluster_to_label: object mapping cluster id (as string) -> label
- label_descriptions: object mapping label -> one-sentence description
"""

        body = {
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": 1600,
            "temperature": LLM_TEMPERATURE,
            "messages": [{"role": "user", "content": prompt}],
        }
        llm_client = get_bedrock_client(profile_name=AWS_PROFILE, region=AWS_REGION)
        response = llm_client.invoke_model(
            modelId=LLM_MODEL_ID,
            body=json.dumps(body),
            contentType="application/json",
            accept="application/json",
        )
        payload = json.loads(response["body"].read().decode("utf-8"))
        return json.loads(payload["content"][0]["text"])

    label_suggestions_l4 = suggest_cluster_labels_with_bedrock(cluster_keywords_l4, "L4")
    label_suggestions_l5 = suggest_cluster_labels_with_bedrock(cluster_keywords_l5, "L5 exploratory")

    with (out_dir / "llm_label_suggestions_l4.json").open("w", encoding="utf-8") as f:
        json.dump(label_suggestions_l4, f, indent=2)
    with (out_dir / "llm_label_suggestions_l5.json").open("w", encoding="utf-8") as f:
        json.dump(label_suggestions_l5, f, indent=2)

    print("Saved LLM naming outputs:")
    print("-", out_dir / "llm_label_suggestions_l4.json")
    print("-", out_dir / "llm_label_suggestions_l5.json")

    print("\nL4 naming suggestions:")
    display(label_suggestions_l4)

    print("\nL5 naming suggestions:")
    display(label_suggestions_l5)
else:
    print("Set RUN_LLM_LABELING=True to generate suggested L4 and L5 category names.")